In [1]:
import numpy as np 
import pandas as pd 


C:\Users\krishna saxena\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df = pd.read_csv("IMDB Dataset.csv")


In [3]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.shape


(50000, 2)

In [6]:
df.drop_duplicates(inplace = True)

In [7]:
df.shape


(49582, 2)

In [8]:
#converting to lower case
df["reviews"] = df["review"].str.lower

In [9]:
#removing urls
import re 

  

In [10]:
def remove_urls(text):
    text = re.sub(r"http\s+","",text)
    return text
df["review"] = df["review"].apply(remove_urls)

In [11]:
df["review"] = df["review"].astype(str)


In [12]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [13]:
def rem_html(text):
    text = re.sub(r"<.*?>" , "",text)
    return text
df["review"] = df["review"].apply(rem_html)

In [14]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\krishna
[nltk_data]     saxena\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\krishna
[nltk_data]     saxena\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\krishna
[nltk_data]     saxena\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [16]:
def rem_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text

df["review"] = df["review"].apply(rem_stopwords)

In [17]:
df.head()


,review,sentiment,reviews
0,One reviewers ntied wtchg 1 Oz epode ll...,positive,<bound method StringMethods.lower of <pandas.c...
1,A wderful ltle producti br br T filming techni...,positive,<bound method StringMethods.lower of <pandas.c...
2,I thought wderful wy spend time o hot su...,positive,<bound method StringMethods.lower of <pandas.c...
3,Bsclly res fmly lttle boy Jke thks res zom...,negative,<bound method StringMethods.lower of <pandas.c...
4,Petter Mtte Love Time Mey vully stunng fi...,positive,<bound method StringMethods.lower of <pandas.c...


In [18]:
#stemming
from nltk.stem import PorterStemmer

In [19]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)
    

In [20]:
#encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [21]:
#vectorisation(text data inti number) using tf-idf

In [22]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

# fit vectorizer
X = tf.fit_transform(df["review"])

# save the CORRECT object
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(tf, f)   # ✅ FIXED

print("Vectorizer saved correctly ✅")

Vectorizer saved correctly ✅


In [23]:
y = df["sentiment"]

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [26]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [27]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

C:\Users\krishna saxena\AppData\Local\Temp\ipykernel_21168\2931448922.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()


In [28]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [29]:
import torch.nn as nn
import torch.optim as optim

In [30]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [31]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [32]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.2478242963552475
epoch = 2/10 and loss = 0.1652950495481491
epoch = 3/10 and loss = 0.1852884739637375
epoch = 4/10 and loss = 0.20181380212306976
epoch = 5/10 and loss = 0.26773950457572937
epoch = 6/10 and loss = 0.3300454616546631
epoch = 7/10 and loss = 0.2921864986419678
epoch = 8/10 and loss = 0.20351402461528778
epoch = 9/10 and loss = 0.342339426279068
epoch = 10/10 and loss = 0.34081530570983887


In [33]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 86.34667742260764


In [34]:


torch.save(model.state_dict(), "model.pth")

In [35]:
 X_train.shape[1]

5000